# 02 Hypothesis Testing

This notebook is executed and includes outputs plus interpretation notes.

## Goal
Test whether monthly patterns, OSB exposure, and sector/severity associations are statistically meaningful.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
sys.path.append(str(Path.cwd().parent))
clean = pd.read_excel('../data/processed/kmo_incidents_clean.xlsx')
istanbul = pd.read_excel('../data/processed/istanbul_enriched.xlsx')
with open('../reports/hypothesis_results.json', encoding='utf-8') as f:
    results = json.load(f)
print(json.dumps(results, indent=2, ensure_ascii=False))

{
  "H1_seasonality_kruskal": {
    "H": 11.810444267496177,
    "p_value": 0.3780589571696683,
    "epsilon_squared": 0.009883466676782643
  },
  "H2_osb_exposure_mannwhitney": {
    "U": 10508.5,
    "p_value": 1.6958186779755807e-29,
    "osb_province_median": 10.0,
    "non_osb_province_median": 1.0
  },
  "H3_osb_exposure_correlations": {
    "osb_parcels_spearman": 0.8068081633360046,
    "osb_parcels_p_value": 1.1733298233032158e-17,
    "osb_area_spearman": 0.7638258632410394,
    "osb_area_p_value": 6.0217020296155325e-15
  },
  "H3b_national_weather_incident_frequency": {
    "temperature_2m_mean_spearman": 0.034355457080020135,
    "temperature_2m_mean_p_value": 1.760811322685956e-57,
    "relative_humidity_2m_mean_spearman": 0.008562202218886225,
    "relative_humidity_2m_mean_p_value": 6.848125748046883e-05,
    "windspeed_10m_max_spearman": 0.047522180801227075,
    "windspeed_10m_max_p_value": 2.5141818581827024e-108,
    "precipitation_sum_spearman": -0.0109906771845616

In [2]:
monthly = clean.groupby(['year','month']).size().reset_index(name='count')
monthly.pivot(index='month', columns='year', values='count').fillna(0).astype(int)

year,2017,2018,2019,2020,2021,2022,2023,2024
month,,,,,,,,
1,11,38,42,48,27,35,31,61
2,8,33,44,26,22,42,41,45
3,0,28,54,37,30,41,36,40
4,0,36,34,29,26,35,27,50
5,12,29,58,37,28,44,36,69
6,19,29,53,32,37,65,50,66
7,15,46,52,42,49,55,71,79
8,6,39,43,39,31,64,62,75
9,23,40,49,61,45,55,49,33


### H1 Interpretation
The Kruskal-Wallis seasonality test is not significant after adding 2024. This means the current evidence does not support a strong month-level seasonal difference in reported industrial incidents across years. The temporal story is weaker than the spatial/exposure story.

In [3]:
province_counts = clean.groupby('il').agg(
    incidents=('Tarih','count'),
    osb_parcels=('osb_parcels','first'),
    osb_area_hectare=('osb_area_hectare','first'),
    has_osb=('has_city_osb_exposure','first')
).sort_values('incidents', ascending=False)
province_counts.head(20)

,incidents,osb_parcels,osb_area_hectare,has_osb
il,,,,
İstanbul,996,1863,2417.60,True
İzmir,399,3487,5070.78,True
Bursa,257,2688,5723.07,True
Kocaeli,224,2129,4032.00,True
Tekirdağ,188,2170,5633.82,True
Ankara,114,14155,9173.46,True
Sakarya,87,1119,2678.70,True
Denizli,77,510,1347.95,True
Adana,72,927,4119.04,True


### H2/H3 Interpretation
The OSB exposure tests are strong: provinces with OSB exposure have higher incident counts, and incident counts correlate strongly with OSB parcel and area measures. This does not mean OSBs cause fires; it means industrial capacity is a major exposure denominator and must be controlled for.

In [4]:
# NATIONAL_WEATHER_HYPOTHESIS
weather_results = pd.Series(results['H3b_national_weather_incident_frequency']).rename('value').to_frame()
weather_results

,value
temperature_2m_mean_spearman,3.435546e-02
temperature_2m_mean_p_value,1.760811e-57
relative_humidity_2m_mean_spearman,8.562202e-03
relative_humidity_2m_mean_p_value,6.848126e-05
windspeed_10m_max_spearman,4.752218e-02
windspeed_10m_max_p_value,2.514182e-108
precipitation_sum_spearman,-1.099068e-02
precipitation_sum_p_value,3.207548e-07
extreme_heat_mean_incidents,1.982826e-02
not_extreme_heat_mean_incidents,1.504648e-02


### H3b Weather Interpretation
The national weather test detects statistically significant relationships because the province-day panel is large, but the effect sizes are small. Mean temperature and wind speed are positively associated with daily province incident frequency, while precipitation is slightly negative. This supports treating weather as a weak contextual temporal covariate, not as the main explanation for industrial incidents.

In [5]:
sector_severity = pd.crosstab(clean['sektor_std'], clean['severity'])
sector_severity

severity,high,low,medium
sektor_std,,,
"Ağaç,Kağıt,Mobilya",21,752,45
Bilinmeyen,9,662,20
Boya,4,39,7
Enerji,5,56,4
Gıda,14,309,18
"Kauçuk,Plastik",7,359,16
"Kozmetik,Temizlik",2,40,5
Metal,41,505,54
"Petrokimya,Yağ",11,111,13


### H4 Interpretation
Sector and severity are statistically associated. This is substantively useful because sectors differ in materials, processes, equipment, and storage conditions. However, it should be interpreted carefully because unknown/missing causes and reporting bias remain important limitations.